# Banking loan default — decision tree

**Target:** `loan_default` (1 = default, 0 = no default). These are 100 synthetic practice records. Place the attached CSV beside this notebook. Run cells in order.

All three notebooks use the same 80/20 split (`random_state=42`). Accuracy on just 20 test records moves in steps of 5 percentage points. Hyperparameters are chosen on training data only; test accuracy may rise or fall.

## 1. Read the CSV and inspect the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay, classification_report)

df = pd.read_csv("Banking_Loan_Default_Classification.csv")
display(df.head())
print("Rows and columns:", df.shape)
print("Missing values:", df.isna().sum().sum())
display(df.describe(include="all").T)

## 2. Check class balance

Count defaults and nondefaults. A 56/44 split is fairly balanced; a model predicting only the majority class would get about 56% accuracy on the full dataset.

In [ ]:
print(df["loan_default"].value_counts())
print((df["loan_default"].value_counts(normalize=True) * 100).round(1))
df["loan_default"].value_counts().sort_index().plot.bar(rot=0, title="0 = No default, 1 = Default")
plt.ylabel("Number of applicants")
plt.show()

## 3. One-hot encoding and fixed train/test split

One-hot encoding turns each text category into a 0/1 column. The target is excluded from the inputs. We hold back 20 records for final evaluation and reuse precisely these rows in all three notebooks.

In [ ]:
X = pd.get_dummies(df.drop(columns="loan_default"), dtype=int)
y = df["loan_default"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Original input columns:", df.shape[1] - 1)
print("Encoded input columns:", X.shape[1])
print("Training records:", len(X_train), "Test records:", len(X_test))
display(X_train.head())

## 4. Model 1 and model 2 for comparison

In [ ]:
basic_model = DecisionTreeClassifier(random_state=42)
basic_model.fit(X_train, y_train)
tuned_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)
tuned_model.fit(X_train, y_train)

## 5. Model 3: GridSearchCV on training rows only

GridSearchCV tries combinations of splitting rule, maximum depth, minimum leaf size, and minimum split size. Four-fold stratified CV assesses candidates within the 80 training rows. The 20 test rows are not involved in selecting the parameters.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

parameter_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 4, 5],
    "min_samples_leaf": [1, 2, 5],
    "min_samples_split": [2, 6]
}
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
search = GridSearchCV(
    DecisionTreeClassifier(random_state=42), parameter_grid,
    scoring="accuracy", cv=cv, refit=True, n_jobs=-1
)
search.fit(X_train, y_train)
print("Best CV accuracy (training folds):", round(search.best_score_, 3))
print("Selected parameters:", search.best_params_)
model = search.best_estimator_

### Which parameter sets performed best during cross-validation?

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
display(cv_results[["mean_test_score", "std_test_score", "params"]]
        .sort_values("mean_test_score", ascending=False).head(10).round(3))
plt.figure(figsize=(20, 8))
plot_tree(model, feature_names=X.columns, class_names=["No default", "Default"],
          filled=True, rounded=True, fontsize=8)
plt.title("GridSearchCV selected decision tree")
plt.show()

## Evaluate the GridSearchCV selected tree

The confusion matrix shows **TN**, **FP**, **FN**, and **TP** (rows = actual, columns = predicted). Accuracy is the fraction correctly classified. ROC AUC measures how well default probabilities rank default cases above nondefaults across thresholds; it is different from accuracy at one threshold.

In [ ]:
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]
print("Training accuracy:", round(accuracy_score(y_train, model.predict(X_train)), 3))
print("Test accuracy:", round(accuracy_score(y_test, pred), 3))
print("Test ROC AUC:", round(roc_auc_score(y_test, prob), 3))
print(classification_report(y_test, pred, target_names=["No default", "Default"], zero_division=0))
display(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0, 1]),
    index=["Actual no default", "Actual default"],
    columns=["Predicted no default", "Predicted default"]))
ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0, 1], display_labels=["No default", "Default"])
plt.show()
RocCurveDisplay.from_predictions(y_test, prob)
plt.show()

### Interpreting this run

Compare training and test accuracy. A large gap can suggest overfitting. On only 20 test cases, one changed prediction shifts accuracy by 5 percentage points. Never choose parameters based on this test score.

## Compare all three trees

The cross-validation score is used for selecting model 3. The test accuracy is reported once for each teaching example. Grid search does **not** promise a higher test score; selecting among models using this test set would make the comparison biased.

In [ ]:
models = {"Basic": basic_model, "Manual hyperparameters": tuned_model,
          "GridSearchCV selected": model}
rows = []
for name, fitted_model in models.items():
    rows.append({
        "Model": name,
        "Train accuracy": fitted_model.score(X_train, y_train),
        "Test accuracy": fitted_model.score(X_test, y_test),
        "Test ROC AUC": roc_auc_score(y_test, fitted_model.predict_proba(X_test)[:, 1])
    })
display(pd.DataFrame(rows).round(3))